[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/templates/30_cosine_lr.ipynb)

# 🟡 Medium: Cosine LR Scheduler with Warmup

*Training*
Implement the learning-rate schedule essentially every modern transformer is
trained with: **linear warmup, then cosine decay**.

$$
\eta(t) =
\begin{cases}
\eta_{\max}\dfrac{t}{T_w} & t < T_w \\[2ex]
\eta_{\min} + \tfrac12(\eta_{\max}-\eta_{\min})
\left(1 + \cos\left(\pi\dfrac{t-T_w}{T-T_w}\right)\right) & t \ge T_w
\end{cases}
$$

### Signature
```python
def cosine_lr_schedule(step, total_steps, warmup_steps, max_lr, min_lr=0.0):
    ...
```

Note the argument order — `total_steps` comes **before** `warmup_steps` and
`max_lr`.

### Rules
- Do not use `optax.warmup_cosine_decay_schedule`
- `step` may be a scalar **or an array** of steps, so branch with `jnp.where`,
  not a Python `if`
- Must be `jax.jit`-able with `step` traced
- Past `total_steps` the rate stays clamped at `min_lr`

### Boundary conventions this is graded on
- $\eta(0) = 0$
- $\eta(T_w) = \eta_{\max}$ exactly — warmup ends *at* the peak
- $\eta(T) = \eta_{\min}$ exactly
- the halfway point of decay is $(\eta_{\max}+\eta_{\min})/2$

### Why warmup exists
At step 0 Adam's second-moment estimate has seen exactly one gradient, so
$\hat{m}/\sqrt{\hat{v}}$ is an unreliable direction with magnitude pinned near 1.
Taking full-size steps in a badly-estimated direction is how early training
diverges, and the deeper the network the worse it gets. Warmup buys the moment
estimates time to become meaningful.

Cosine decay then matters at the other end: it holds a high rate for a long
time and anneals smoothly to near zero, which beats step decay empirically and,
unlike a linear ramp, does not waste the final steps at a rate too small to
make progress.

### A JAX note
The PyTorch original branches with a Python `if` and returns a float, which is
fine for a scalar step. Writing it with `jnp.where` instead costs nothing, and
buys you a schedule that works on a whole array of steps at once and survives
`jit` when the step is traced.

In [ ]:
# Install jax-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q jax-judge flax')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

import jax
import jax.numpy as jnp


def cosine_lr_schedule(step, total_steps, warmup_steps, max_lr, min_lr=0.0):
    """Learning rate at `step` — linear warmup then cosine decay.

    Args:
        step:         scalar or array of step indices
        total_steps:  step at which the rate reaches min_lr
        warmup_steps: length of the linear ramp
        max_lr:       peak rate, reached at step == warmup_steps
        min_lr:       floor

    Returns:
        Learning rate(s), same shape as `step`.
    """
    pass  # Replace this

In [ ]:
# 🔍 Scratch cell — poke at your implementation
import jax.numpy as jnp

steps = jnp.arange(0, 1001, 100)
lrs = cosine_lr_schedule(steps, total_steps=1000, warmup_steps=100, max_lr=1e-3)
for s, lr in zip(steps.tolist(), lrs.tolist()):
    bar = "█" * int(lr / 1e-3 * 40)
    print(f"{s:>5}  {lr:.6f}  {bar}")

In [ ]:
# ✅ SUBMIT — run this cell to check your solution
from jax_judge import check, hint, solution

check("cosine_lr")

# hint("cosine_lr")      # stuck? nudge without the answer
# solution("cosine_lr")  # spoiler: the reference implementation